# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [9]:
%idle_timeout 2880
%glue_version 4.0
%worker_type G.1X
%number_of_workers 2

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

You are already connected to a glueetl session edcf95e7-f15b-415c-ac0d-717f2af72fe9.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Current idle_timeout is 2880 minutes.
idle_timeout has been set to 2880 minutes.


You are already connected to a glueetl session edcf95e7-f15b-415c-ac0d-717f2af72fe9.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Setting Glue version to: 4.0


You are already connected to a glueetl session edcf95e7-f15b-415c-ac0d-717f2af72fe9.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous worker type: G.1X
Setting new worker type to: G.1X


You are already connected to a glueetl session edcf95e7-f15b-415c-ac0d-717f2af72fe9.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous number of workers: 5
Setting new number of workers to: 2



In [10]:
dyf = glueContext.create_dynamic_frame_from_catalog(
    database='lds_raw',
    table_name='medidor'
)

df = dyf.toDF()
df.show(10)
df.printSchema()


+----------+-------------+-------------+------------------+------------+-----------------+------------+--------------+
|id_medidor|id_suministro|marca_medidor|tecnologia_medidor|numero_serie|fecha_instalacion|fecha_retiro|estado_medidor|
+----------+-------------+-------------+------------------+------------+-----------------+------------+--------------+
|   2000000|      1000000|   Landis+Gyr|       Electrónico|  LAN-843XXX|       2024-12-30|            |        ACTIVO|
|   2000001|      1000001|      Actaris|       Electrónico|  ACT-816XXX|       2024-08-13|            |        ACTIVO|
|   2000002|      1000002|       Elster|               AMI|  ELS-536XXX|       2024-02-08|  2024-12-31|      RETIRADO|
|   2000003|      1000002|       Holley|       Electrónico|  HOL-527XXX|       2024-12-31|  2024-12-31|      RETIRADO|
|   2000004|      1000002|       Elster|       Electrónico|  ELS-871XXX|       2024-12-31|            |        ACTIVO|
|   2000005|      1000003|        Iskra|       E

In [11]:
total = df.count()
print("Total filas:", total)


Total filas: 13855


In [12]:
import pyspark.sql.functions as F

print("=== NULOS POR COLUMNA ===")
df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()


=== NULOS POR COLUMNA ===
+----------+-------------+-------------+------------------+------------+-----------------+------------+--------------+
|id_medidor|id_suministro|marca_medidor|tecnologia_medidor|numero_serie|fecha_instalacion|fecha_retiro|estado_medidor|
+----------+-------------+-------------+------------------+------------+-----------------+------------+--------------+
|         0|            0|            0|                 0|           0|                0|           0|             0|
+----------+-------------+-------------+------------------+------------+-----------------+------------+--------------+


In [13]:
string_cols = [c for c, t in df.dtypes if t == "string"]

print("=== BLANCOS POR COLUMNA ===")
df.select([
    F.count(F.when(F.col(c) == "", c)).alias(c)
    for c in string_cols
]).show()


=== BLANCOS POR COLUMNA ===
+-------------+------------------+------------+-----------------+------------+--------------+
|marca_medidor|tecnologia_medidor|numero_serie|fecha_instalacion|fecha_retiro|estado_medidor|
+-------------+------------------+------------+-----------------+------------+--------------+
|            0|                 0|           0|                0|       11653|             0|
+-------------+------------------+------------+-----------------+------------+--------------+


In [14]:
print("=== MEDIDORES CON FECHA_RETIRO < FECHA_INSTALACION ===")
df.filter(F.col("fecha_retiro") < F.col("fecha_instalacion")).show(10)


=== MEDIDORES CON FECHA_RETIRO < FECHA_INSTALACION ===
+----------+-------------+-------------+------------------+------------+-----------------+------------+--------------+
|id_medidor|id_suministro|marca_medidor|tecnologia_medidor|numero_serie|fecha_instalacion|fecha_retiro|estado_medidor|
+----------+-------------+-------------+------------------+------------+-----------------+------------+--------------+
|   2000000|      1000000|   Landis+Gyr|       Electrónico|  LAN-843XXX|       2024-12-30|            |        ACTIVO|
|   2000001|      1000001|      Actaris|       Electrónico|  ACT-816XXX|       2024-08-13|            |        ACTIVO|
|   2000004|      1000002|       Elster|       Electrónico|  ELS-871XXX|       2024-12-31|            |        ACTIVO|
|   2000005|      1000003|        Iskra|       Electrónico|  ISK-799XXX|       2023-08-03|            |        ACTIVO|
|   2000006|      1000004|        Itron|               AMI|  ITR-943XXX|       2024-09-21|            |        A

In [15]:
df.select(
    F.min("fecha_instalacion").alias("min_instalacion"),
    F.max("fecha_instalacion").alias("max_instalacion")
).show()



+---------------+---------------+
|min_instalacion|max_instalacion|
+---------------+---------------+
|     2018-06-19|     2024-12-31|
+---------------+---------------+


In [8]:
df.groupBy("id_cliente").count().orderBy(F.col("count").desc()).show(10)


+----------+-----+
|id_cliente|count|
+----------+-----+
|      1219|    3|
|       510|    3|
|      2068|    3|
|       224|    3|
|      2766|    3|
|      4992|    3|
|      5800|    3|
|       232|    3|
|      6973|    3|
|      4201|    3|
+----------+-----+
only showing top 10 rows
